# ขั้นที่ 4: ทดลอง TTS ภาษาไทย 1 คลิป
เปิด notebook นี้แยกจาก EDA เพื่ออ่านรายการ Train ที่บันทึกไว้แล้ว ไม่ต้องรัน EDA ใหม่

หัวข้อ 1–4 ทดลองคลิปแรก; หัวข้อ 5–6 ขยายเป็นตัวอย่าง Train รวม 10 คลิปสำหรับฟังตรวจ ไม่ใช่การสร้างทั้งชุด Train/Dev

ใช้ **MMS Thai** ที่ฝึกมาแล้ว สร้างเสียงสังเคราะห์จากข้อความ **Train** หนึ่งรายการ ไม่ train TTS และไม่โคลนเสียงผู้พูดต้นทาง เสียงสังเคราะห์ถือเป็น `spoof` สำหรับงานตรวจจับของเรา แต่ยังไม่เพิ่มเข้าชุด train จนกว่าจะฟังตรวจ

เลือก Kernel `.venv` แล้วรันจากบนลงล่าง ถ้าเครื่องใหม่ยังไม่มีแพ็กเกจ: `python -m pip install -e ".[eda,tts]"` จาก project root; ไม่ติดตั้งแพ็กเกจอัตโนมัติใน notebook

[Model card MMS Thai](https://huggingface.co/facebook/mms-tts-tha) · [วิธีใช้ VITS](https://huggingface.co/docs/transformers/model_doc/vits)
โมเดลมีเงื่อนไข **CC-BY-NC 4.0** ต้องตรวจสิทธิ์ก่อนใช้เชิงพาณิชย์ เราโหลดเฉพาะ safetensors/config/tokenizer/README ที่ revision คงที่ ไม่โหลดโค้ด remote หรือ weights pickle

ไฟล์โมเดลอยู่ `checkpoints/tts/` ส่วนเสียงตัวอย่างและรายงานอยู่ `data/processed/cvtts/pilot_v1/` ทั้งสองไม่เข้า Git ไม่มีการส่งเสียงคนไปยัง API สร้างเสียง; รันโมเดลในเครื่องด้วย CPU

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd
from IPython.display import Audio, display

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("เปิด notebook ภายในโฟลเดอร์ project ก่อน")
    PROJECT_ROOT = PROJECT_ROOT.parent

PILOT_DIR = PROJECT_ROOT / "data/processed/cvtts/pilot_v1"
TRAIN_MANIFEST = PILOT_DIR / "manifests/real_train.csv"
split_report = json.loads((PILOT_DIR / "qc/pilot_split_report.json").read_text(encoding="utf-8"))
assert hashlib.sha256(TRAIN_MANIFEST.read_bytes()).hexdigest() == split_report["manifest_sha256"]["real_train.csv"]

df_train = pd.read_csv(TRAIN_MANIFEST, dtype="string", keep_default_na=False)
assert len(df_train) == 80 and df_train["split"].eq("train").all()
# รายการเรียง sample_id ไว้แล้ว; เลือกหนึ่งข้อความจาก Train เท่านั้น
source_row = df_train.sort_values("sample_id").iloc[0]
assert source_row["label"] == "bonafide"
tts_text = source_row["sentence"]
assert tts_text.strip()
real_audio_path = (PROJECT_ROOT / source_row["audio_path"]).resolve()
assert real_audio_path.is_relative_to((PROJECT_ROOT / "data/raw/common_voice").resolve())
assert hashlib.sha256(real_audio_path.read_bytes()).hexdigest() == source_row["audio_file_sha256"]

print("source sample:", source_row["sample_id"])
print("split:", source_row["split"])
print("ข้อความที่จะให้ TTS อ่าน:", tts_text)
df_train[["sample_id", "sentence", "split"]].head()

## 1. โหลดโมเดลที่ฝึกมาแล้ว
ดาวน์โหลดครั้งแรกประมาณ 145 MB สำหรับ weights; รอบต่อไปอ่านจากไฟล์ในเครื่อง การโหลด checkpoint ไม่ใช่การ train

ล็อก revision และ seed เพื่อบันทึกว่าตัวอย่างมาจากโมเดล/การตั้งค่าใด ค่า `noise_scale` ของ VITS คือความสุ่มภายในตัวสร้างเสียง **ไม่ใช่ environmental noise** ที่จะใช้ในการเปรียบเทียบ Clean/Mixed ภายหลัง

In [ ]:
import torch
import transformers
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, VitsModel

MODEL_ID = "facebook/mms-tts-tha"
MODEL_REVISION = "fcecc37a91566f4a36ba6c6c8aa39830ff6daa9d"
MODEL_DIR = PROJECT_ROOT / "checkpoints/tts/mms-tts-tha" / MODEL_REVISION
MODEL_FILES = [
    "model.safetensors", "config.json", "tokenizer_config.json",
    "special_tokens_map.json", "vocab.json", "README.md",
]
# ถ้ามีไฟล์ครบแล้ว ไม่เรียกเครือข่ายซ้ำ
if not all((MODEL_DIR / filename).is_file() for filename in MODEL_FILES):
    snapshot_download(
        repo_id=MODEL_ID, revision=MODEL_REVISION,
        local_dir=MODEL_DIR, allow_patterns=MODEL_FILES,
    )

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR, local_files_only=True, trust_remote_code=False,
)
model = VitsModel.from_pretrained(
    MODEL_DIR, local_files_only=True, use_safetensors=True,
).to("cpu").eval()
print("model:", MODEL_ID)
print("revision:", MODEL_REVISION)
print("native sample rate:", model.config.sampling_rate)
print("tokenizer requires uroman:", tokenizer.is_uroman)
print("parameters:", sum(parameter.numel() for parameter in model.parameters()))

## 2. สร้าง waveform หนึ่งคลิป
ใช้ข้อความเท่านั้น ไม่ป้อนเสียงคนต้นทางให้ TTS ความยาวของเสียงสร้างใหม่ขึ้นกับข้อความและโมเดล ไม่บังคับ 4.04 วินาทีในขั้นนี้

ตรวจว่า waveform ไม่ว่าง เป็นค่าตัวเลขทั้งหมด และไม่เงียบเกือบศูนย์ นี่เป็นการตรวจทางเทคนิค **ไม่ใช่การรับรองว่าออกเสียงถูก** ต้องฟังตรวจเอง

In [ ]:
import numpy as np
import time

TTS_SEED = 42
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
torch.manual_seed(TTS_SEED)
inputs = tokenizer(tts_text, return_tensors="pt")
assert inputs["input_ids"].numel() > 0
if tokenizer.unk_token_id is not None:
    unknown_count = int((inputs["input_ids"] == tokenizer.unk_token_id).sum())
    assert unknown_count == 0, "ข้อความมี token ที่ไม่รองรับ ต้องตรวจ tokenizer ก่อน"
else:
    unknown_count = None

started = time.perf_counter()
with torch.inference_mode():
    waveform = model(**inputs).waveform.squeeze(0).cpu().numpy().astype("float32")
generation_seconds = time.perf_counter() - started
sample_rate = int(model.config.sampling_rate)

assert waveform.ndim == 1 and waveform.size > 0
assert np.isfinite(waveform).all()
assert float(np.abs(waveform).max()) > 1e-8
print("ความยาวเสียงที่สร้าง:", round(len(waveform) / sample_rate, 3), "วินาที")
print("sample rate:", sample_rate, "Hz")
print("เวลาที่ใช้สร้างบน CPU:", round(generation_seconds, 3), "วินาที")
print("peak:", float(np.abs(waveform).max()))

## 3. บันทึกเสียงและที่มา
บันทึก WAV float32 ที่ native sample rate โดยไม่ normalize/crop/เติม noise หรือเปลี่ยนไฟล์เสียงคน รายงานบอก seed, revision, package versions, model/file hashes และข้อความต้นทาง

รายงาน `manual_review.status = pending`: ไม่ประกาศว่าคุณภาพผ่านก่อนคนฟัง เสียงนี้สืบทอด split = Train จากข้อความต้นทาง; `source_sample_id` คือ lineage ไม่ใช่ตัวตนของผู้พูด TTS

ถ้าตัวอย่าง/รายงานเดิมมีเนื้อหาต่างกัน โค้ดหยุดก่อนเขียนทับ ใช้ชื่อ experiment/version ใหม่เมื่อตั้งใจเปลี่ยนการตั้งค่า

In [ ]:
import io
import importlib.metadata as metadata
from scipy.io import wavfile

fake_id = f"mms_tha_{source_row['sample_id']}_seed{TTS_SEED}"
FAKE_PATH = PILOT_DIR / "native_tts/mms_tha/train" / f"{fake_id}.wav"
REPORT_PATH = PILOT_DIR / "qc/tts_samples" / f"{fake_id}.json"

# scipy เขียน float32 WAV ลง memory ก่อนตรวจเนื้อหา ไม่ clip ค่าหรือแปลง PCM16
wav_buffer = io.BytesIO()
wavfile.write(wav_buffer, sample_rate, waveform)
wav_bytes = wav_buffer.getvalue()
report = {
    "sample_id": fake_id,
    "audio_path": FAKE_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "label": "spoof", "split": "train", "language": "th",
    "source_sample_id": source_row["sample_id"],
    "source_train_manifest_sha256": hashlib.sha256(TRAIN_MANIFEST.read_bytes()).hexdigest(),
    "source_audio_file_sha256": source_row["audio_file_sha256"],
    "source_text_key": source_row["text_key"], "tts_text": tts_text,
    "generator_model": MODEL_ID, "generator_revision": MODEL_REVISION,
    "generator_license": "CC-BY-NC-4.0",
    "generation_seed": TTS_SEED, "device": "cpu", "cpu_threads": 2,
    "deterministic_algorithms": True,
    "settings": {
        "speaking_rate": model.config.speaking_rate,
        "noise_scale": model.config.noise_scale,
        "noise_scale_duration": model.config.noise_scale_duration,
    },
    "tokenizer_is_uroman": tokenizer.is_uroman,
    "unknown_token_count": unknown_count,
    "sample_rate": sample_rate, "channels": 1,
    "samples": int(waveform.size), "duration_s": waveform.size / sample_rate,
    "wav_subtype": "FLOAT", "peak": float(np.abs(waveform).max()),
    "audio_file_sha256": hashlib.sha256(wav_bytes).hexdigest(),
    "waveform_sha256": hashlib.sha256(waveform.astype("<f4", copy=False).tobytes()).hexdigest(),
    "model_file_sha256": {
        filename: hashlib.sha256((MODEL_DIR / filename).read_bytes()).hexdigest()
        for filename in MODEL_FILES
    },
    "software": {
        package: metadata.version(package)
        for package in ["torch", "transformers", "tokenizers", "safetensors", "huggingface-hub", "numpy", "scipy"]
    },
    "manual_review": {"status": "pending", "text_matches": None, "notes": ""},
}
prepared_files = {
    FAKE_PATH: wav_bytes,
    REPORT_PATH: (json.dumps(report, ensure_ascii=False, indent=2) + "\n").encode("utf-8"),
}
for path, content in prepared_files.items():
    if path.exists() and path.read_bytes() != content:
        raise FileExistsError(f"ตัวอย่างเดิมมีเนื้อหาต่างกัน ไม่เขียนทับ: {path}")
for path, content in prepared_files.items():
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        path.write_bytes(content)
    print(path.relative_to(PROJECT_ROOT).as_posix())

## 4. ฟังเทียบข้อความ
ฟังเสียงคนและเสียง TTS จากข้อความเดียวกัน ตรวจว่าอ่านตรงคำ ครบประโยค เป็นภาษาไทย และไม่เงียบ/วน/ขาดตอน
ไม่จำเป็นต้องเหมือนเสียงคนต้นทาง เพราะเราไม่ได้ทำ voice cloning

ขั้นต่อไปหลังฟังผ่านจึงค่อยสร้างเพิ่ม โดยทุกไฟล์ต้องมี source lineage และ split เดิม ยังไม่ใช้คะแนน AASIST/RawNet2 มาคัดให้เหลือแต่เสียงที่ตรวจง่าย

In [ ]:
print("ข้อความ:", tts_text)
print("เสียงจริงจาก Common Voice:")
display(Audio(filename=str(real_audio_path)))
print("เสียงสร้างใหม่จาก MMS Thai:")
display(Audio(filename=str(FAKE_PATH)))
print("ผลฟังตรวจล่าสุด: ดูหัวข้อ 6 และไฟล์ qc/tts_reviews")

## 5. เพิ่มตัวอย่างจาก Train ให้ครบ 10 ข้อความ
เก็บตัวอย่างเดิมไว้ 1 ข้อความ แล้วเลือกอีก 9 ข้อความจาก Train ให้กระจายตั้งแต่ข้อความสั้นถึงยาวตามจำนวนตัวอักษร ไม่ใช้ข้อมูล Dev หรือคะแนน detector มาช่วยเลือก

ใช้โมเดลและค่าการสร้างเดิม พร้อม seed 42 แยกต่อคลิป ไม่ clone เสียงมนุษย์ รูปแบบเสียง/สำเนียงของ TTS ไม่สืบทอดอายุ เพศ หรือสำเนียงจากเจ้าของเสียงจริง

คลิปเดิมที่คุณแจ้งว่าอ่านตรงคำและชัดบันทึกผลฟังไว้แยกใน `qc/tts_reviews/` โดยไม่แก้ provenance การสร้างเดิม ส่วนอีก 9 คลิปยังรอฟังตรวจ หากไฟล์เดิมมีอยู่จะตรวจ hash/metadata ก่อนใช้ต่อ ไม่เขียนทับไฟล์ที่ไม่ตรงกัน

In [ ]:
# เลือกข้อความที่มีความยาวหลากหลาย ไม่ใช่สุ่มเฉพาะแถวแรก 10 แถว
remaining_train = df_train[df_train["sample_id"].ne(source_row["sample_id"])].copy()
remaining_train["text_chars"] = remaining_train["sentence"].str.len()
remaining_train = remaining_train.sort_values(["text_chars", "sample_id"]).reset_index(drop=True)
positions = np.linspace(0, len(remaining_train) - 1, 9).round().astype(int)
assert len(set(positions)) == 9

df_batch = pd.concat(
    [source_row.to_frame().T, remaining_train.iloc[positions]],
    ignore_index=True,
)
df_batch["text_chars"] = df_batch["sentence"].str.len()
assert len(df_batch) == 10 and df_batch["sample_id"].is_unique
assert df_batch["split"].eq("train").all()
df_batch[["sample_id", "sentence", "text_chars", "split"]]

In [ ]:
import copy
import soundfile as sf

batch_reports = []
for _, row in df_batch.iterrows():
    text = row["sentence"]
    original_path = (PROJECT_ROOT / row["audio_path"]).resolve()
    assert original_path.is_relative_to((PROJECT_ROOT / "data/raw/common_voice").resolve())
    assert hashlib.sha256(original_path.read_bytes()).hexdigest() == row["audio_file_sha256"]

    sample_id = f"mms_tha_{row['sample_id']}_seed{TTS_SEED}"
    wav_path = PILOT_DIR / "native_tts/mms_tha/train" / f"{sample_id}.wav"
    json_path = PILOT_DIR / "qc/tts_samples" / f"{sample_id}.json"
    assert wav_path.exists() == json_path.exists(), "พบไฟล์คู่เสียง/รายงานไม่ครบ ต้องตรวจเองก่อน"

    batch_inputs = tokenizer(text, return_tensors="pt")
    assert batch_inputs["input_ids"].numel() > 0
    batch_unknown_count = (
        int((batch_inputs["input_ids"] == tokenizer.unk_token_id).sum())
        if tokenizer.unk_token_id is not None else None
    )
    assert batch_unknown_count in (0, None), f"ข้อความมี token ที่ไม่รองรับ: {row['sample_id']}"

    # ที่มาและค่าตั้งต้นร่วมกัน แต่ข้อมูลเฉพาะคลิปต้องเปลี่ยนให้ตรงทุกช่อง
    expected = copy.deepcopy(report)
    expected.update({
        "sample_id": sample_id,
        "audio_path": wav_path.relative_to(PROJECT_ROOT).as_posix(),
        "source_sample_id": row["sample_id"],
        "source_audio_file_sha256": row["audio_file_sha256"],
        "source_text_key": row["text_key"],
        "tts_text": text,
        "unknown_token_count": batch_unknown_count,
    })

    if wav_path.exists():
        saved = json.loads(json_path.read_text(encoding="utf-8"))
        audio, native_sr = sf.read(wav_path, dtype="float32")
        assert audio.ndim == 1 and audio.size > 0 and np.isfinite(audio).all()
        assert native_sr == sample_rate and float(np.abs(audio).max()) > 1e-8
        # ตรวจ provenance ทั้งหมด รวม waveform ที่อ่านกลับ ไม่ดูเฉพาะชื่อไฟล์
        expected.update({
            "samples": int(audio.size), "duration_s": audio.size / native_sr,
            "peak": float(np.abs(audio).max()),
            "audio_file_sha256": hashlib.sha256(wav_path.read_bytes()).hexdigest(),
            "waveform_sha256": hashlib.sha256(audio.astype("<f4", copy=False).tobytes()).hexdigest(),
        })
        assert saved == expected, f"ไฟล์เก่าไม่ตรง provenance ปัจจุบัน: {sample_id}"
        sample_report = saved
        action = "ใช้ไฟล์เดิมที่ตรวจแล้ว"
    else:
        torch.manual_seed(TTS_SEED)  # reset ต่อคลิป เพื่อไม่ขึ้นกับลำดับหรือ resume
        with torch.inference_mode():
            audio = model(**batch_inputs).waveform.squeeze(0).cpu().numpy().astype("float32")
        assert audio.ndim == 1 and audio.size > 0 and np.isfinite(audio).all()
        assert float(np.abs(audio).max()) > 1e-8
        audio_buffer = io.BytesIO()
        wavfile.write(audio_buffer, sample_rate, audio)
        content = audio_buffer.getvalue()
        expected.update({
            "samples": int(audio.size), "duration_s": audio.size / sample_rate,
            "peak": float(np.abs(audio).max()),
            "audio_file_sha256": hashlib.sha256(content).hexdigest(),
            "waveform_sha256": hashlib.sha256(audio.astype("<f4", copy=False).tobytes()).hexdigest(),
        })
        sample_report = expected
        wav_path.parent.mkdir(parents=True, exist_ok=True)
        json_path.parent.mkdir(parents=True, exist_ok=True)
        # exclusive create: ไม่ทับไฟล์ที่อาจถูกสร้างระหว่างรัน
        with wav_path.open("xb") as handle:
            handle.write(content)
        with json_path.open("xb") as handle:
            handle.write((json.dumps(sample_report, ensure_ascii=False, indent=2) + "\n").encode("utf-8"))
        action = "สร้างใหม่"

    batch_reports.append(sample_report)
    print(f"{len(batch_reports):02d}/10 | {action} | {audio.size / sample_rate:.2f}s | {text[:60]}")

assert len(batch_reports) == 10
df_tts_batch = pd.DataFrame([
    {key: item[key] for key in [
        "sample_id", "source_sample_id", "audio_path", "tts_text",
        "split", "label", "sample_rate", "channels", "duration_s", "audio_file_sha256",
    ]}
    for item in batch_reports
])
assert df_tts_batch["source_sample_id"].is_unique
assert df_tts_batch["audio_file_sha256"].is_unique
BATCH_MANIFEST = PILOT_DIR / "manifests/tts_train_review10.csv"
batch_content = df_tts_batch.to_csv(index=False, lineterminator="\n", float_format="%.10g").encode("utf-8")
if BATCH_MANIFEST.exists():
    assert BATCH_MANIFEST.read_bytes() == batch_content, "รายการ review10 เปลี่ยน ต้องใช้ version ใหม่"
else:
    with BATCH_MANIFEST.open("xb") as handle:
        handle.write(batch_content)
print("บันทึกรายการตัวอย่าง:", BATCH_MANIFEST.relative_to(PROJECT_ROOT).as_posix())
df_tts_batch[["source_sample_id", "tts_text", "duration_s"]]

## 6. ฟังตรวจตัวอย่าง 10 คลิป
ฟังทีละข้อ เทียบกับข้อความที่แสดง กดเสียงจริงเพิ่มได้ถ้าต้องการตรวจคำอ่านต้นทาง

คำตอบที่ต้องการ: อ่านตรง/ครบหรือไม่ ชัดหรือไม่ มีเสียงวนหรือขาดตอนหรือไม่ ไม่ใช้ “ฟังเหมือน AI” เป็นเหตุผลเดียวในการทิ้งคลิป และไม่คัดด้วยคะแนน detector

ผลคลิปแรกที่คุณแจ้งไว้ใช้เป็นการฟังตรวจเบื้องต้นเท่านั้น ไม่ใช่ผลประเมินโมเดล อีก 9 คลิปยัง `pending` หากมีคำผิดให้บอกหมายเลขคลิป ไม่แก้ CSV/ข้อความดิบหรือ provenance เดิมทับโดยไม่บันทึก

In [ ]:
for number, item in enumerate(batch_reports, start=1):
    review_path = PILOT_DIR / "qc/tts_reviews" / f"{item['sample_id']}.json"
    review = json.loads(review_path.read_text(encoding="utf-8")) if review_path.is_file() else None
    if review is not None:
        assert review["audio_file_sha256"] == item["audio_file_sha256"], "ผลฟังตรวจอ้างถึงเสียงคนละไฟล์"
    status = review["status"] if review is not None else "pending"
    print(f"คลิป {number}: {item['tts_text']}")
    print(f"ความยาว {item['duration_s']:.2f} วินาที | ผลฟังตรวจ: {status}")
    display(Audio(filename=str(PROJECT_ROOT / item["audio_path"])))

## 7. ทดลองแก้ตัว “ำ” เฉพาะคลิป 10
จากการฟังของผู้ใช้ คลิป 10 มีคำอ่านขาด; ตรวจพบว่า tokenizer เดิมตัด `ำ` (U+0E33) ออกจริง ขณะที่รองรับ `ํ` (U+0E4D) และ `า` (U+0E32)

ทดลองเปลี่ยนเฉพาะรูปอักษร `ำ → ํ + า` ก่อนเข้า tokenizer โดยเก็บข้อความเดิมไว้แยก ไม่แก้คำ/ความหมาย และตรวจว่า tokenizer ไม่ตัดตัวอักษรหลังเตรียม นี่เป็นการแก้ข้อผิดพลาดของ input ยัง **ไม่รับรอง** ว่าออกเสียงถูกทุกคำ

ใช้ seed 42 โมเดล CPU และค่าการสร้างเดิม เปลี่ยนเฉพาะการเตรียมข้อความ บันทึกเสียงใหม่ชื่อลงท้าย `sara_am_v1_seed42` ไม่ทับคลิป 10 เดิมหรือ manifest review10 และยังไม่แก้คลิป 3 ไม่สร้างเสียงจาก Dev

หัวข้อ 1–6 คงไว้เป็น baseline เดิมสำหรับเทียบข้อผิดพลาด อย่าใช้สร้างชุดเต็มจนกว่าจะตกลง text-preparation policy ใหม่

In [ ]:
from thai_spoof.cvtts.text import (
    MMS_THAI_TEXT_VERSION,
    prepare_mms_thai_text,
    require_preserved_tokenizer_text,
)

am_row = df_batch.iloc[9]
assert am_row["split"] == "train"
original_text = am_row["sentence"]
fixed_text = prepare_mms_thai_text(original_text)
require_preserved_tokenizer_text(fixed_text, tokenizer)
fixed_inputs = tokenizer(fixed_text, return_tensors="pt")
fixed_tokens = tokenizer.convert_ids_to_tokens(fixed_inputs["input_ids"][0].tolist())
assert tokenizer.convert_tokens_to_string(fixed_tokens) == fixed_text
fixed_unknown_count = (
    int((fixed_inputs["input_ids"] == tokenizer.unk_token_id).sum())
    if tokenizer.unk_token_id is not None else None
)
assert fixed_unknown_count in (0, None)

old_prepared_text, _ = tokenizer.prepare_for_tokenization(original_text)
assert old_prepared_text != original_text, "การทดลองนี้ต้องอ้างถึงข้อผิดพลาดเดิม"
print("ข้อความต้นฉบับ:", original_text)
print("ข้อความที่เข้าโมเดลเดิม:", old_prepared_text)
print("ข้อความที่เข้าโมเดลหลังเตรียมใหม่:", fixed_text)
print("tokenizer เก็บข้อความใหม่ครบ: ผ่าน")
print("text preparation:", MMS_THAI_TEXT_VERSION)

In [ ]:
# โหลด provenance คลิป 10 เดิม ไม่แก้ข้อมูลเก่า
old_id = f"mms_tha_{am_row['sample_id']}_seed{TTS_SEED}"
old_report_path = PILOT_DIR / "qc/tts_samples" / f"{old_id}.json"
old_report = json.loads(old_report_path.read_text(encoding="utf-8"))
old_wav_path = PROJECT_ROOT / old_report["audio_path"]
assert hashlib.sha256(old_wav_path.read_bytes()).hexdigest() == old_report["audio_file_sha256"]
assert old_report["tts_text"] == original_text
assert old_report["generator_revision"] == MODEL_REVISION
assert old_report["generation_seed"] == TTS_SEED

torch.manual_seed(TTS_SEED)
with torch.inference_mode():
    fixed_audio = model(**fixed_inputs).waveform.squeeze(0).cpu().numpy().astype("float32")
assert fixed_audio.ndim == 1 and fixed_audio.size > 0 and np.isfinite(fixed_audio).all()
assert float(np.abs(fixed_audio).max()) > 1e-8

fixed_id = f"mms_tha_{am_row['sample_id']}_sara_am_v1_seed{TTS_SEED}"
fixed_wav_path = PILOT_DIR / "native_tts/mms_tha/train" / f"{fixed_id}.wav"
fixed_report_path = PILOT_DIR / "qc/tts_samples" / f"{fixed_id}.json"
fixed_buffer = io.BytesIO()
wavfile.write(fixed_buffer, sample_rate, fixed_audio)
fixed_wav_bytes = fixed_buffer.getvalue()

fixed_report = copy.deepcopy(old_report)
fixed_report.update({
    "sample_id": fixed_id,
    "audio_path": fixed_wav_path.relative_to(PROJECT_ROOT).as_posix(),
    "parent_sample_id": old_id,
    "parent_audio_file_sha256": old_report["audio_file_sha256"],
    "experiment_reason": "tokenizer_dropped_U+0E33_in_baseline",
    "original_text": original_text,
    "tts_text": fixed_text,
    "text_preparation_version": MMS_THAI_TEXT_VERSION,
    "text_preparation_rule": "U+0E33_to_U+0E4D_U+0E32_only",
    "tokenizer_preserves_prepared_text": True,
    "token_roundtrip_verified": True,
    "unknown_token_count": fixed_unknown_count,
    "samples": int(fixed_audio.size),
    "duration_s": fixed_audio.size / sample_rate,
    "peak": float(np.abs(fixed_audio).max()),
    "audio_file_sha256": hashlib.sha256(fixed_wav_bytes).hexdigest(),
    "waveform_sha256": hashlib.sha256(fixed_audio.astype("<f4", copy=False).tobytes()).hexdigest(),
    "manual_review": {"status": "pending", "text_matches": None, "notes": ""},
})
fixed_files = {
    fixed_wav_path: fixed_wav_bytes,
    fixed_report_path: (json.dumps(fixed_report, ensure_ascii=False, indent=2) + "\n").encode("utf-8"),
}
for path, content in fixed_files.items():
    if path.exists() and path.read_bytes() != content:
        raise FileExistsError(f"ผลการทดลองเปลี่ยน ไม่เขียนทับ: {path}")
for path, content in fixed_files.items():
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        with path.open("xb") as handle:
            handle.write(content)
    print(path.relative_to(PROJECT_ROOT).as_posix())
print("ความยาวใหม่:", round(fixed_audio.size / sample_rate, 3), "วินาที")
print("ไม่แทนที่คลิปเดิม; ผลฟังตรวจล่าสุดดู qc/tts_reviews และหัวข้อ 8")

## 8. ฟังคลิป 10 ก่อน–หลังเตรียม “ำ”
ฟังคำว่า **ชำระเงิน** ทั้งสองตำแหน่ง และตรวจว่าคำอื่นยังครบด้วย หากดีขึ้นบางส่วนแต่ยังมีคำหาย ให้บอกตรงนั้น ไม่ถือว่าทั้งคลิปผ่านอัตโนมัติ
คลิป 3 ยังเป็น `needs_review` และไม่ถูกเปลี่ยนในขั้นนี้

In [ ]:
print("ข้อความ:", original_text)
print("ก่อนแก้ — คลิป 10 เดิม:")
display(Audio(filename=str(old_wav_path)))
am_review_path = PILOT_DIR / "qc/tts_reviews" / f"{fixed_id}.json"
am_review = json.loads(am_review_path.read_text(encoding="utf-8")) if am_review_path.exists() else None
if am_review is not None:
    assert am_review["audio_file_sha256"] == fixed_report["audio_file_sha256"]
print("หลังเตรียมรูปอักษร ำ ใหม่ — ผลฟังตรวจ:", am_review["status"] if am_review else "pending")
display(Audio(filename=str(fixed_wav_path)))

## 9. ทดลองคลิป 3 ด้วย seed อีก 3 ค่า
ผู้ใช้ได้ยินคำว่า **โม้** ในเสียงเดิมเป็น **โหมด** แต่ตรวจแล้ว tokenizer เก็บข้อความครบ จึงทดลองเฉพาะความสุ่มการสร้างเสียง

กำหนด seeds **7, 123, 2026** ก่อนฟังผล เทียบกับ baseline seed 42 โดยคงข้อความ โมเดล revision ค่า speaking rate/noise scales และ CPU เหมือนเดิม ไม่เปลี่ยนคำเป็นคำสะกดหลอกและไม่ใช้คะแนน detector คัดเสียง

ตัวเตรียม `ำ` ที่เพิ่มไว้ไม่เปลี่ยนข้อความนี้ จึงตรวจว่า model input เหมือน baseline จริง บันทึกเสียงทั้ง 3 แบบพร้อมรายงานแยก ไม่ลบ/แทนคลิปเดิม ไม่เปลี่ยน manifest review10 และยังไม่ใช้ทั้ง 3 แบบเป็นตัวอย่างอิสระในชุด train

ทั้ง 3 คลิปยังรอฟังตรวจ ถ้าเกิดคำผิดซ้ำทุกแบบ ไม่สรุปว่าเพิ่ม seed แล้วแก้ได้ ต้องพิจารณา TTS ตัวอื่นหรือทบทวนขอบเขตคุณภาพก่อนสร้างชุดใหญ่

In [ ]:
# ใช้รหัสคลิปที่รายงานว่ามีปัญหา ไม่เลือกตัวอย่างจากคะแนนโมเดล
probe_matches = df_train[df_train["sample_id"].eq("common_voice_th_25669007")]
assert len(probe_matches) == 1
probe_row = probe_matches.iloc[0]
assert probe_row["split"] == "train"
probe_text = prepare_mms_thai_text(probe_row["sentence"])
assert probe_text == probe_row["sentence"], "ขั้นนี้เปลี่ยนได้เฉพาะ seed ไม่ใช่ข้อความ"
require_preserved_tokenizer_text(probe_text, tokenizer)
probe_inputs = tokenizer(probe_text, return_tensors="pt")
probe_tokens = tokenizer.convert_ids_to_tokens(probe_inputs["input_ids"][0].tolist())
assert tokenizer.convert_tokens_to_string(probe_tokens) == probe_text

PROBE_SEEDS = [7, 123, 2026]
assert len(set(PROBE_SEEDS)) == 3 and TTS_SEED not in PROBE_SEEDS
probe_parent_id = f"mms_tha_{probe_row['sample_id']}_seed{TTS_SEED}"
probe_parent_report_path = PILOT_DIR / "qc/tts_samples" / f"{probe_parent_id}.json"
probe_parent = json.loads(probe_parent_report_path.read_text(encoding="utf-8"))
probe_parent_wav = PROJECT_ROOT / probe_parent["audio_path"]
assert hashlib.sha256(probe_parent_wav.read_bytes()).hexdigest() == probe_parent["audio_file_sha256"]
assert probe_parent["tts_text"] == probe_text
assert probe_parent["generator_revision"] == MODEL_REVISION
assert probe_parent["settings"] == {
    "speaking_rate": model.config.speaking_rate,
    "noise_scale": model.config.noise_scale,
    "noise_scale_duration": model.config.noise_scale_duration,
}
print("ข้อความเหมือนเดิม:", probe_text)
print("baseline seed:", TTS_SEED)
print("ทดลอง seeds:", PROBE_SEEDS)

In [ ]:
probe_reports = []
for probe_seed in PROBE_SEEDS:
    torch.manual_seed(probe_seed)
    with torch.inference_mode():
        probe_audio = model(**probe_inputs).waveform.squeeze(0).cpu().numpy().astype("float32")
    assert probe_audio.ndim == 1 and probe_audio.size > 0 and np.isfinite(probe_audio).all()
    assert float(np.abs(probe_audio).max()) > 1e-8
    probe_id = f"mms_tha_{probe_row['sample_id']}_seed_probe_v1_seed{probe_seed}"
    probe_wav_path = PILOT_DIR / "native_tts/mms_tha/train" / f"{probe_id}.wav"
    probe_report_path = PILOT_DIR / "qc/tts_samples" / f"{probe_id}.json"
    probe_buffer = io.BytesIO()
    wavfile.write(probe_buffer, sample_rate, probe_audio)
    probe_wav_bytes = probe_buffer.getvalue()

    probe_report = copy.deepcopy(probe_parent)
    probe_report.update({
        "sample_id": probe_id,
        "audio_path": probe_wav_path.relative_to(PROJECT_ROOT).as_posix(),
        "parent_sample_id": probe_parent_id,
        "parent_audio_file_sha256": probe_parent["audio_file_sha256"],
        "experiment": "clip3_seed_probe_v1",
        "experiment_variable": "generation_seed",
        "predeclared_probe_seeds": PROBE_SEEDS,
        "original_text": probe_row["sentence"],
        "text_preparation_version": MMS_THAI_TEXT_VERSION,
        "tokenizer_preserves_prepared_text": True,
        "token_roundtrip_verified": True,
        "generation_seed": probe_seed,
        "samples": int(probe_audio.size),
        "duration_s": probe_audio.size / sample_rate,
        "peak": float(np.abs(probe_audio).max()),
        "audio_file_sha256": hashlib.sha256(probe_wav_bytes).hexdigest(),
        "waveform_sha256": hashlib.sha256(probe_audio.astype("<f4", copy=False).tobytes()).hexdigest(),
        "manual_review": {"status": "pending", "text_matches": None, "notes": ""},
    })
    probe_files = {
        probe_wav_path: probe_wav_bytes,
        probe_report_path: (json.dumps(probe_report, ensure_ascii=False, indent=2) + "\n").encode("utf-8"),
    }
    for path, content in probe_files.items():
        if path.exists() and path.read_bytes() != content:
            raise FileExistsError(f"ผล seed probe เปลี่ยน ไม่เขียนทับ: {path}")
    for path, content in probe_files.items():
        path.parent.mkdir(parents=True, exist_ok=True)
        if not path.exists():
            with path.open("xb") as handle:
                handle.write(content)
    probe_reports.append(probe_report)
    print(f"seed {probe_seed}: {probe_report['duration_s']:.3f} วินาที | pending")

probe_summary = {
    "experiment": "clip3_seed_probe_v1",
    "source_sample_id": probe_row["sample_id"],
    "split": "train",
    "tts_text": probe_text,
    "baseline_sample_id": probe_parent_id,
    "baseline_audio_file_sha256": probe_parent["audio_file_sha256"],
    "baseline_seed": TTS_SEED,
    "probe_seeds": PROBE_SEEDS,
    "samples": [
        {key: item[key] for key in ["sample_id", "generation_seed", "audio_path", "audio_file_sha256", "duration_s"]}
        for item in probe_reports
    ],
    "selection": "not_selected_pending_human_review",
    "purpose": "pronunciation_diagnostic_not_detector_evaluation",
}
probe_summary_path = PILOT_DIR / "qc/tts_experiments/clip3_seed_probe_v1.json"
probe_summary_bytes = (json.dumps(probe_summary, ensure_ascii=False, indent=2) + "\n").encode("utf-8")
if probe_summary_path.exists():
    assert probe_summary_path.read_bytes() == probe_summary_bytes, "ผลทดลองเปลี่ยน ต้องใช้ชื่อ experiment ใหม่"
else:
    probe_summary_path.parent.mkdir(parents=True, exist_ok=True)
    with probe_summary_path.open("xb") as handle:
        handle.write(probe_summary_bytes)
pd.DataFrame(probe_summary["samples"])[["generation_seed", "duration_s", "sample_id"]]

## 10. ฟังคำว่า “โม้”: baseline และ 3 seeds
ฟังเสียงจริงจาก Common Voice เป็นตัวอ้างอิงคำอ่าน ตามด้วยเสียงเดิม seed 42 และตัวอย่าง A/B/C ระบุว่าแต่ละแบบอ่าน **โม้** ถูกหรือยังได้ยินเป็น **โหมด** หรือคำอื่น พร้อมตรวจว่าคำอื่นในประโยคไม่หาย

นี่เป็นการเปรียบเทียบการออกเสียง ไม่ได้พิสูจน์ว่าตัวตรวจจับเรียนรู้ดีขึ้น และไม่ได้กำหนดนโยบายเลือก seed ของ dataset เต็ม

In [ ]:
print("ข้อความ:", probe_text)
print("เสียงจริงจาก Common Voice:")
display(Audio(filename=str(PROJECT_ROOT / probe_row["audio_path"])))
print("baseline — seed 42 (คุณแจ้งว่าได้ยินคล้าย โหมด):")
display(Audio(filename=str(probe_parent_wav)))
for label, item in zip(["A", "B", "C"], probe_reports):
    review_path = PILOT_DIR / "qc/tts_reviews" / f"{item['sample_id']}.json"
    review = json.loads(review_path.read_text(encoding="utf-8")) if review_path.exists() else None
    if review is not None:
        assert review["audio_file_sha256"] == item["audio_file_sha256"]
    status = review["status"] if review else "pending"
    print(f"แบบ {label} — seed {item['generation_seed']} | ผลฟังตรวจ: {status}")
    display(Audio(filename=str(PROJECT_ROOT / item["audio_path"])))